01 · Data cleaning and exploratory analysis

Research question: How accurately can concrete compressive strength be predicted from mixture composition and curing age?

Dataset: UCI Machine Learning Repository, Concrete Compressive Strength (Dataset 165, DOI 10.24432/C5PK67), 1,030 samples, 8 predictors, 1 continuous target (MPa).

This notebook follows the six cleaning steps from our presentation, produces the descriptive statistics and correlations, and saves a cleaned dataset to `data/concrete_clean.csv` for `02_modelling.ipynb`.

1. Load the data

The loader looks for the data in this order:
1. `data/concrete.csv`, a CSV copy of the UCI file
2. `data/Concrete_Data.xls`, the original UCI file (needs `xlrd`)
3. If neither is found, it downloads the dataset with the `ucimlrepo` package and saves `data/concrete.csv`.

Columns are renamed **by position** because the UCI headers are long and are not identical across versions of the file.

In [ ]:
SHORT_NAMES = ["cement", "slag", "fly_ash", "water", "superplasticizer",
               "coarse_agg", "fine_agg", "age", "strength"]

def load_raw():
    xls, csv = DATA_DIR / "Concrete_Data.xls", DATA_DIR / "concrete.csv"
    if csv.exists():
        df = pd.read_csv(csv)
        source = csv.name
    elif xls.exists():
        df = pd.read_excel(xls)                      # needs: pip install xlrd
        source = xls.name
    else:
        from ucimlrepo import fetch_ucirepo          # pip install ucimlrepo
        ds = fetch_ucirepo(id=165)
        df = pd.concat([ds.data.features, ds.data.targets], axis=1)
        DATA_DIR.mkdir(exist_ok=True)
        df.to_csv(csv, index=False)
        source = "ucimlrepo (saved to data/concrete.csv)"
    print(f"Loaded {df.shape[0]} rows x {df.shape[1]} columns from {source}")
    return df

raw = load_raw()
raw.head()

2. Step 1: standardise column names

In [ ]:
assert raw.shape[1] == 9, f"Expected 9 columns, found {raw.shape[1]}"
original_names = list(raw.columns)
df = raw.copy()
df.columns = SHORT_NAMES
pd.DataFrame({"original": original_names, "short": SHORT_NAMES})

3. Step 2: check data types

In [ ]:
print(df.dtypes)
non_numeric = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
assert not non_numeric, f"Non-numeric columns: {non_numeric}"
print("\nAll columns are numeric ✔")

4. Step 3: check missing values

UCI reports none, so we confirm that after import.

In [ ]:
missing = df.isna().sum()
print(missing)
print(f"\nTotal missing values: {missing.sum()}")

5. Step 4: check duplicates

Exact duplicate rows have identical mix, age and strength. Keeping them could put the same sample in both the train and test set, which would make test performance look better than it really is. They carry no new information, so we remove them and record how many were removed.

In [ ]:
n_dup = df.duplicated().sum()
print(f"Exact duplicate rows: {n_dup}")
display(df[df.duplicated(keep=False)].sort_values(SHORT_NAMES).head(10))

df = df.drop_duplicates().reset_index(drop=True)
print(f"Rows after removing duplicates: {len(df)}")

6. Step 5: inspect ranges and outliers

We flag outliers with the 1.5 × IQR rule but do not delete them. An extreme value can be a real mix design (for example a 365-day test or a high-superplasticiser mix). Physically impossible values, such as negative quantities, would be removed.

In [ ]:
assert (df >= 0).all().all(), "Negative values found, check the data"
assert (df["age"] >= 1).all(), "Age must be at least 1 day"

q1, q3 = df.quantile(0.25), df.quantile(0.75)
iqr = q3 - q1
outlier_counts = ((df < q1 - 1.5 * iqr) | (df > q3 + 1.5 * iqr)).sum()
print("IQR-flagged values per column (kept, not removed):")
print(outlier_counts)

fig, axes = plt.subplots(3, 3, figsize=(12, 9))
for ax, col in zip(axes.ravel(), df.columns):
    ax.boxplot(df[col], widths=0.5)
    ax.set_title(col)
    ax.set_xticks([])
fig.suptitle("Boxplots of all variables (outliers flagged, not removed)")
fig.tight_layout()
fig.savefig(FIG_DIR / "01_boxplots.png", bbox_inches="tight")
plt.show()

7. Step 6: confirm units

Ingredients are in kg/m³, age in days and strength in MPa. A sanity check is that a cubic metre of concrete should weigh about 2,200 to 2,600 kg.

In [ ]:
UNITS = {c: "kg/m³" for c in SHORT_NAMES[:7]} | {"age": "days", "strength": "MPa"}
total_mass = df[SHORT_NAMES[:7]].sum(axis=1)
print(total_mass.describe().round(1))
print("\nUnits:", UNITS)

8. Descriptive statistics

In [ ]:
desc = df.describe().T[["mean", "std", "min", "50%", "max"]].round(2)
desc.insert(0, "unit", [UNITS[c] for c in desc.index])
desc.to_csv(RESULTS_DIR / "descriptive_statistics.csv")
desc

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(12, 9))
for ax, col in zip(axes.ravel(), df.columns):
    ax.hist(df[col], bins=30, color="#4C72B0", edgecolor="white")
    ax.set_title(f"{col} ({UNITS[col]})")
fig.suptitle("Distributions of all variables")
fig.tight_layout()
fig.savefig(FIG_DIR / "01_histograms.png", bbox_inches="tight")
plt.show()

zero_share = (df[["slag", "fly_ash", "superplasticizer"]] == 0).mean().mul(100).round(1)
print("% of mixes with zero of each supplementary material:")
print(zero_share)

9. Correlation with strength

In [ ]:
corr = df.corr(method="pearson")
corr_target = corr["strength"].drop("strength").sort_values(ascending=False)
corr_target.round(3).to_csv(RESULTS_DIR / "correlation_with_strength.csv")
print(corr_target.round(3))

fig, ax = plt.subplots(figsize=(8, 6.5))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)), corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr)), corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8,
                color="white" if abs(corr.iloc[i, j]) > 0.5 else "black")
fig.colorbar(im, ax=ax, shrink=0.8, label="Pearson r")
ax.set_title("Correlation matrix")
fig.tight_layout()
fig.savefig(FIG_DIR / "01_correlation_matrix.png", bbox_inches="tight")
plt.show()

In [ ]:
# Scatter plots of strength against each predictor
fig, axes = plt.subplots(2, 4, figsize=(15, 7), sharey=True)
for ax, col in zip(axes.ravel(), SHORT_NAMES[:8]):
    ax.scatter(df[col], df["strength"], s=8, alpha=0.5)
    ax.set_xlabel(f"{col} ({UNITS[col]})")
    ax.set_title(f"r = {corr.loc[col, 'strength']:.2f}")
for ax in axes[:, 0]:
    ax.set_ylabel("strength (MPa)")
fig.suptitle("Strength vs each predictor: no single feature explains strength")
fig.tight_layout()
fig.savefig(FIG_DIR / "01_scatter_vs_strength.png", bbox_inches="tight")
plt.show()

In [ ]:
# Strength rises with age, and the rise is clearly nonlinear
age_profile = df.groupby("age")["strength"].agg(["mean", "count"]).round(2)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(age_profile.index, age_profile["mean"], marker="o")
ax.set_xscale("log")
ax.set_xlabel("age (days, log scale)")
ax.set_ylabel("mean strength (MPa)")
ax.set_title("Mean strength by curing age")
fig.tight_layout()
fig.savefig(FIG_DIR / "01_strength_by_age.png", bbox_inches="tight")
plt.show()
age_profile

10. Save the cleaned dataset

In [ ]:
clean_path = DATA_DIR / "concrete_clean.csv"
df.to_csv(clean_path, index=False)
print(f"Saved {len(df)} rows to {clean_path.relative_to(ROOT)}")

Summary

* The data has 9 numeric columns and no missing values. Exact duplicate rows were removed.
* Outliers were flagged but kept, because they are plausible mix designs.
* No single predictor is strongly correlated with strength. Cement has the strongest positive correlation, followed by superplasticiser and age, and water is negative.
* Strength rises with age in a clearly nonlinear way, which motivates comparing a linear baseline with nonlinear models in `02_modelling.ipynb`.